# Simple Kaggle baseline (persistence)

This notebook stays self-contained on Kaggle: it uses only the official competition dataset and writes `submission.csv` into `/kaggle/working` so it can be submitted without any extra inputs.  The baseline repeats each player's last observed position for every forecasted frame.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path('/kaggle/input/nfl-big-data-bowl-2026-prediction')
if not DATA_ROOT.exists():
    alt = Path('/kaggle/input/nfl-big-data-bowl-2026')
    if alt.exists():
        DATA_ROOT = alt
    else:
        raise FileNotFoundError(
            'Attach the official competition dataset (nfl-big-data-bowl-2026-prediction) when creating the Kaggle notebook.'
        )
print('Using data root:', DATA_ROOT)

In [ ]:
sample_path = DATA_ROOT / 'sample_submission.csv'
test_input_path = DATA_ROOT / 'test_input.csv'

sample_df = pd.read_csv(sample_path)
test_input = pd.read_csv(test_input_path)
print('sample rows:', len(sample_df))
print('test_input rows:', len(test_input))
test_input.head()

In [ ]:
group_cols = ['game_id', 'play_id', 'nfl_id']
last_obs = (
    test_input.sort_values('frame_id')
    .groupby(group_cols, as_index=False)
    .tail(1)[group_cols + ['x', 'y']]
)

parts = sample_df['id'].str.split('_', expand=True)
parts.columns = ['game_id', 'play_id', 'nfl_id', 'frame_id']
parts = parts.astype(int)
sample_df = pd.concat([parts, sample_df.drop(columns='id')], axis=1)

submission = sample_df.merge(last_obs, on=group_cols, how='left', suffixes=('', '_last'))
submission['x'] = submission['x_last'].fillna(0.0)
submission['y'] = submission['y_last'].fillna(0.0)
submission = submission.drop(columns=['x_last', 'y_last'])
submission['id'] = (
    submission['game_id'].astype(str)
    + '_' + submission['play_id'].astype(str)
    + '_' + submission['nfl_id'].astype(str)
    + '_' + submission['frame_id'].astype(str)
)
submission = submission[['id', 'x', 'y']]

output_path = Path('/kaggle/working/submission.csv')
submission.to_csv(output_path, index=False)
print('Wrote submission to', output_path)
print('submission rows:', len(submission))
submission.head()